## Intro to POC Mode

[POC mode](https://nvflare.readthedocs.io/en/main/user_guide/poc_command.html) allows users to test the features of a full FLARE deployment on a single machine, without the overhead of a true distributed deployment and the need to establish secure communication between server and client systems.

Compared to the FL Simulator, where the job run is automated on a single system, POC mode allows you to establish and connect distinct server and client "systems" which can then be orchestrated using the FLARE Console.  This can be useful in preparation for a distributed deployment.

To get started, let's look at the NVFlare CLI usage for the poc subcommand:

In [ ]:
!nvflare poc -h

### Preparing the POC environment
Before running POC mode, there are a couple important environment variables that should be set.

First, to simplify deploying the example apps in the NVFlare GitHub repo, you can set `NVFLARE_HOME` to the root of the GitHub clone.  In this case, we've cloned to our current working directory, so we can set it as:

In [ ]:
import os
workdir=os.getcwd()
%env NVFLARE_HOME={workdir}/../NVFlare

By default, POC mode uses a temporary workspace in /tmp/nvflare/poc.  We would like to keep the workspace within our working directory, so let's create a poc_workspace dir.  We can then use the `NVFLARE_POC_WORKSPACE` variable to define this as the POC workspace.

Note:  if you have previously created the poc_workspace, you will want to clean it up using the `nvflare poc --clean` command or manually remove the `poc_workspace` directory.

In [ ]:
# !nvflare poc --clean
!rm -r poc_workspace
!mkdir poc_workspace
%env NVFLARE_POC_WORKSPACE={workdir}/poc_workspace

### Preparing the POC workspace

Now that we've configured out POC environment, we can prepare the POC workspace.  By default, this will generate POC packages for a server and two clients.

(Note that `nvflare poc --prepare` prompts you to create the workspace.)

In [ ]:
!printf '%s\n' y | nvflare poc --prepare

Let's take a look.

In [ ]:
!tree poc_workspace

### Running the POC Deployment

When starting the POC deployment, it's necessary to use a separate terminal since the `nvflare poc --start` command will run  in the foreground emitting output from the server and any connected clients.

Also note that `nvflare poc --start` starts all participants, including the admin console.  It's often nice to start server and clients separately so that we can interact with the deployment using a separate admin console.  To do this, we'll pass the `-ex admin` arg to exclude the admin client from the initial POC run and use the FLARE API to run admin commands separately.

So pop open the launcher, launch a terminal, and run (remembering to set the NVFLARE_POC_WORKSPACE and NVFLARE_HOME vars!):

```shell
export NVFLARE_POC_WORKSPACE=$(pwd -P)/notebooks/poc_workspace
export NVFLARE_HOME=$(pwd -P)/NVFlare
nvflare poc --start -ex admin
```

Keep this terminal open so you can continue to watch server and client output.

### Using the FLARE API to connect to the POC deployment

The admin directory contains the startup script for the FLARE Console, which can be used interactively to operate a running FLARE deployment.  A FLARE deployment can also be managed using the FLARE API, which will use the configuration in the admin directory to connect to the FLARE server.  Since we already have the server and clients running in the background from the above terminal commands, we'll use FLARE API to start a new admin session and connect.

To get started, we need to import the FLARE API class and initialize session.

In [ ]:
from nvflare.fuel.flare_api.flare_api import new_insecure_session

admin_session = new_insecure_session(startup_kit_location = workdir + "/poc_workspace/admin")
print(admin_session.get_system_info())

### Launching a job with the FLARE API
Next we can use the FLARE API to launch one of the hello-world examples and monitor its status.  Note the difference here as compared to the Simulator example.  Because we're running a POC deployment with unique workspaces for the FLARE server and clients, we don't need to define a local workspace for job results before submitting the job.

All that's required to launch the job is the path to the job configuration.  This configuration is pushed to the server workspace and deployed to clients, and all job results are collected back in the server workspace.  After the job completes, we can use the FLARE API to download the results of the job from the server workspace to our admin directory.


In [ ]:
!tree /flare/NVFlare/examples/hello-world/hello-pt/jobs/hello-pt

In [ ]:
path_to_job_config = "/flare/NVFlare/examples/hello-world/hello-pt/jobs/hello-pt"
job_id = admin_session.submit_job(path_to_job_config)
print("Submitted job with job ID" + job_id)

### Monitoring the state of the FLARE deployment and job status

Now that the job is submitted, we can use the FLARE API to query the state of the system, show job status, and display job metadata.  These capabilities are especially useful through the course of a FLARE experiment, when you typically execute multiple jobs through the course of the course of the experiment.

For example, you can query the state of all jobs (with optional detailed output including job metadata), or query the metadata for a specific job by ID.

In [ ]:
import json

# Job Status
jobs_output = admin_session.list_jobs()
jobs_detail = admin_session.list_jobs(detailed=True)
print("Job Status")
print(json.dumps((jobs_output), indent=2))
print("\nJob Detail")
print(json.dumps((jobs_detail), indent=2))

# Job Metadata
print("\nJob Metadata")
admin_session.get_job_meta(job_id)

### Monitoring a job run with a callback function
You can also construct a simple callback function to monitor job status during a run.

In [ ]:
from nvflare.fuel.flare_api.flare_api import Session

def sample_cb(
        session: Session, job_id: str, job_meta, *cb_args, **cb_kwargs
    ) -> bool:
    if job_meta["status"] == "RUNNING":
        if cb_kwargs["cb_run_counter"]["count"] < 3:
            print(job_meta)
            print(cb_kwargs["cb_run_counter"])
        else:
            print(".", end="")
    else:
        print("\n" + str(job_meta))
    
    cb_kwargs["cb_run_counter"]["count"] += 1
    return True

admin_session.monitor_job(job_id, cb=sample_cb, cb_run_counter={"count":0})

### Stopping the POC deployment (important!)
Once the job has completed, we can stop the server and clients in the POC deployent.  This is necessary to free up ports for the following notebook examples!

In [ ]:
!nvflare poc --stop

### Retrieving job results
When the cell above monitoring job output shows that the job has finished with `<MonitorReturnCode.JOB_FINISHED: 0>`, we can use the FLARE API to download job results.  This is useful when running the FLARE API on a remote deployment and you wish to review job artifacts on your local laptop or workstation.

In [ ]:
job_download = admin_session.download_job_result(job_id)
print("Job download path: " + job_download)

In [ ]:
!tree {job_download}

In [ ]:
!tail {job_download}/workspace/log.txt

In [ ]:
cross_val_file = open(job_download + "/workspace/cross_site_val/cross_val_results.json")
cross_val_json = json.load(cross_val_file)
print(json.dumps(cross_val_json, indent=2))